In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')
CAMINHO_DATA = '/content/drive/MyDrive/Projeto_Futebol/data'

# 3. Modeling and Evaluation (Metodologia CRISP-ML)

## 3.1. Estratégia de Treino e Divisão Temporal (Time-Series Split)
Para respeitar a ordem cronológica dos dados, faz-se uma divisão sequencial sem baralhar (*no shuffle*):
- **80% dos jogos mais antigos:** Conjunto de Treino.
- **20% dos jogos mais recentes:** Conjunto de Teste.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# 1. Carregar dataset preparado
caminho_prep = os.path.join(CAMINHO_DATA, "dataset_futebol_preparado.csv")
df = pd.read_csv(caminho_prep)

# 2. Definir Features e Target
features = [
    'Home_Golos_Marcados_L5', 'Away_Golos_Marcados_L5',
    'Home_Golos_Sofridos_L5', 'Away_Golos_Sofridos_L5',
    'Home_Forma_L5', 'Away_Forma_L5'
]

X = df[features]
y_raw = df['FTR']

# 3. Label Encoding
le = LabelEncoder()
y = le.fit_transform(y_raw)

# 4. Divisão Temporal 80/20
corte = int(len(df) * 0.80)
X_train, X_test = X.iloc[:corte], X.iloc[corte:]
y_train, y_test = y[:corte], y[corte:]

# 5. Normalização para modelos lineares
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"✅ Treino (80%): {X_train.shape[0]} jogos | Teste (20%): {X_test.shape[0]} jogos")

## 3.2. Treino e Comparação dos Algoritmos
Treinam-se e comparam-se os seguintes modelos:
1. **Regressão Logística**
2. **Random Forest Classifier**
3. **Gradient Boosting Classifier**

In [ ]:
modelos = {
    "Regressão Logística": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42, max_depth=6),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42, max_depth=3)
}

resultados = {}
previsoes = {}

print("=" * 70)
print("AVALIAÇÃO DE PERFORMANCE DOS MODELOS")
print("=" * 70 + "\n")

for nome, modelo in modelos.items():
    X_tr = X_train_scaled if nome == "Regressão Logística" else X_train
    X_te = X_test_scaled if nome == "Regressão Logística" else X_test

    modelo.fit(X_tr, y_train)
    y_pred = modelo.predict(X_te)

    previsoes[nome] = y_pred
    acc = accuracy_score(y_test, y_pred)
    resultados[nome] = acc

    print(f"📌 {nome}")
    print(f"Accuracy no Teste: {acc * 100:.2f}%")
    print(classification_report(y_test, y_pred, target_names=le.classes_))
    print("-" * 50)

df_res = pd.DataFrame(list(resultados.items()), columns=['Modelo', 'Accuracy'])
df_res['Accuracy_%'] = df_res['Accuracy'] * 100
print("\n=== QUADRO RESUMO DE ACCURACY ===")
print(df_res.sort_values(by='Accuracy_%', ascending=False).to_string(index=False))

## 3.3. Matrizes de Confusão e Importância das Variáveis
Visualização gráfica dos acertos/erros dos modelos (Matrizes de Confusão) e relevância das métricas pré-jogo (*Feature Importance*).

In [ ]:
# 1. Matrizes de Confusão
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, (nome, y_pred) in enumerate(previsoes.items()):
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=le.classes_, yticklabels=le.classes_, ax=axes[i]
    )
    axes[i].set_title(f"Matriz de Confusão: {nome}")
    axes[i].set_xlabel("Previsão")
    axes[i].set_ylabel("Real")

plt.tight_layout()
plt.show()

# 2. Importância das Features (Random Forest)
rf_model = modelos["Random Forest"]
importancias = pd.Series(rf_model.feature_importances_, index=features).sort_values(ascending=True)

plt.figure(figsize=(10, 6))
importancias.plot(kind='barh', color='teal')
plt.title("Importância das Variáveis Pré-Jogo (Random Forest)")
plt.xlabel("Grau de Importância Relativa")
plt.ylabel("Variável Preditiva")
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()